# 2026-10-19 Lab RNN

**Author:** DILING69 · **Notebook:** lab-rnn · **Created:** 2026-10-19T17:31:12-07:00

**Purpose:** train the pilot's networks on the lab's reviewed blocks (docs/lab-rnn) and report the training checks. The axis analysis is not here: runs are not looked at past the training checks until its spec is approved.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import hashlib
import os

import matplotlib.pyplot as plt
import numpy as np
import scienceplots  # noqa: F401  registers the SciencePlots styles

from lab_rnn import blocks
from lab_rnn.config import LAB, LAB_SMOKE
from pilot import train
from pilot.analysis import THRESHOLDS, load_thresholds
from pilot.config import THRESHOLDS_SHA256

plt.style.use(["science", "notebook", "no-latex"])
PRESET = {"LAB_SMOKE": LAB_SMOKE, "LAB": LAB}[os.environ.get("LAB_PRESET", "LAB_SMOKE")]  # LAB_PRESET=LAB: real run
INPUT_SETS = ("place_only", "place_speed")  # no turn rate or head direction: both come from the measured heading


## 1. Blocks and split

The reviewed blocks as sessions at 20 Hz, split whole by recording day in date order. Training windows start every 25 steps in the arena and every 100 on the track; validation and test windows every 100.

In [ ]:
sessions, cfg = blocks.load(PRESET)  # reads the processed folder only; copies and runs go under LAB_RNN_RUNS
folder = f"{cfg['runs_dir']}/{cfg['name']}_{cfg['data_id']}"
print(f"preset {cfg['name']}, data id {cfg['data_id']}, folder {folder}")

def n_windows(session, stride):
    return len(range(0, len(session.t) - cfg["window"] - cfg["horizon"] + 1, stride))

print(f"{'split':6} {'world':6} {'sessions':>8} {'blocks':>6} {'minutes':>8} {'windows':>8}")
for split in ("train", "val", "test"):
    for world in cfg["worlds"]:
        group = [s for s in sessions[world] if s.split == split]
        stride = cfg["train_stride"][world] if split == "train" else cfg["stride"]
        print(f"{split:6} {world:6} {len({s.session_id for s in group}):8} {len(group):6} "
              f"{sum(len(s.t) for s in group) * cfg['dt'] / 60:8.1f} {sum(n_windows(s, stride) for s in group):8}")
for split in ("train", "val", "test"):
    print(split, sorted({s.session_id for w in sessions.values() for s in w if s.split == split}))


## 2. Training

The pilot's arms and settings (`pilot.train.run_sweep`), on `arena` and `track` in place of `open_field` and `loop`. Complete runs are reused; interrupted and failed runs are kept.

In [ ]:
sha = hashlib.sha256(THRESHOLDS.read_bytes()).hexdigest()
thr = load_thresholds()
print("thresholds.yaml", "matches" if sha == THRESHOLDS_SHA256 else "DOES NOT MATCH", "the pre-registered hash")
jobs = train.sweep_jobs(cfg, INPUT_SETS)
print(f"{len(jobs)} runs:", ", ".join(f"{arm.name} x {len(train.seeds_of(arm, cfg))}"
                                      for arm in dict.fromkeys(arm for arm, _ in jobs)))


In [ ]:
results = train.run_sweep(cfg, sessions, INPUT_SETS)
print(f"{len(results)} of {len(jobs)} runs complete")


In [ ]:
arms = [arm for input_set in INPUT_SETS for arm in train.arms_for(input_set, cfg["diets"]) if arm.kind != "untrained"]
fig, axes = plt.subplots(1, len(arms), figsize=(3.2 * len(arms), 3), sharey=True, squeeze=False)
for ax, arm in zip(axes[0], arms):
    for seed in train.seeds_of(arm, cfg):
        if (arm, seed) in results:
            metrics = results[(arm, seed)]["metrics"]
            epochs = np.arange(1, metrics["epochs"] + 1)
            ax.plot(epochs, metrics["train_loss"], color="C0", lw=1)
            ax.plot(epochs, metrics["val_loss"], color="C1", lw=1)
    ax.set_title(f"{arm.input_set}\n{train.label(arm)}", fontsize=9)
    ax.set_xlabel("epoch")
axes[0][0].set_ylabel("loss (train blue, val orange)")
fig.tight_layout()


## 3. Training checks

The pilot's section 2 checks with its pre-registered thresholds.

In [ ]:
persistence = train.persistence_losses(cfg, sessions)
for world in ("all", *cfg["worlds"]):
    print(f"test loss, {world}")
    print("\n".join(train.loss_table(cfg, results, persistence, INPUT_SETS, world)), end="\n\n")
print("decoder R-squared from hidden state, test sessions (heading is validation only)")
print("\n".join(train.decoder_table(cfg, results, INPUT_SETS)))


In [ ]:
checks = train.section_2_checks(cfg, results, persistence, INPUT_SETS, thr)
for name, passed, numbers in checks:
    print(f"[{'PASS' if passed else 'FAIL'}] {name}\n       {numbers}")
if len(results) == len(jobs):
    blocks.write_summary(cfg, persistence, checks, sha)
    print(f"\nbaselines.json and checks.json in {folder}")
else:
    print(f"\nthe sweep is not finished ({len(results)} of {len(jobs)}): no summary written")


## Notes

- Every run's weights, checkpoints, metrics and manifest are under the folder printed in section 1, with the session
  copies, layouts and `blocks.json`; nothing is written to the processed folder.
- Nothing here looks at axis tuning. That waits for the triple-T analysis spec (docs/lab-rnn/tasks.md, Left).
